In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [6]:
# Load market data
market = pd.read_csv("../../data/processed/market_all_features.csv", parse_dates=["date"])
market["ticker"] = market["ticker"].astype(str)

print("Market shape:", market.shape)
display(market.head())


Market shape: (52454, 21)


,date,open,high,low,close,volume,adj close,ticker,sector,cap,return,log_return,year,vol21,vol63,ret_ema21,ret_ema63,px_ma21,px_ma63,px_ma200,trend200_up
0,2018-01-03,19.562585,19.749109,19.547663,19.719265,5519259.0,19.719265,ABB,Real Estate - Energy - Industrials,Mid Cap,NaN,NaN,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2018-01-04,19.838642,19.958017,19.749111,19.935635,5738092.0,19.935635,ABB,Real Estate - Energy - Industrials,Mid Cap,0.010972,0.010913,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2018-01-05,19.920712,20.084852,19.883406,20.084852,4435594.0,20.084852,ABB,Real Estate - Energy - Industrials,Mid Cap,0.007485,0.007457,2018,NaN,NaN,0.010972,0.010972,NaN,NaN,NaN,0
3,2018-01-08,20.099777,20.137081,19.920715,20.092316,5029780.0,20.092316,ABB,Real Estate - Energy - Industrials,Mid Cap,0.000372,0.000372,2018,NaN,NaN,0.010655,0.010864,NaN,NaN,NaN,0
4,2018-01-09,19.972940,20.196768,19.935636,20.196768,6974533.0,20.196768,ABB,Real Estate - Energy - Industrials,Mid Cap,0.005199,0.005185,2018,NaN,NaN,0.009721,0.010536,NaN,NaN,NaN,0


In [12]:
market.ticker.unique()

array(['ABB', 'Alcon', 'Alpine', 'Aryzta', 'BKW Energie', 'BLKB',
       'Baloise', 'Emmi', 'Givaudan', 'Helvetia', 'Holcim', 'Julius Baer',
       'Landis', 'Logitech', 'Lonza', 'Nestle', 'Novartis', 'Reishauer',
       'Richemont', 'Roche', 'Santhera', 'Swatch', 'Swiss Life',
       'Swiss Prime Site', 'Swiss Re', 'Swisscom', 'Swissquote', 'UBS',
       'Vontobel', 'Zurich Insurance'], dtype=object)

In [4]:
# Load sentiment (Parquet is preferred)
sent = pd.read_parquet("../../data/processed/sentiment_daily.parquet")
sent["date"] = pd.to_datetime(sent["date"])
sent["ticker"] = sent["ticker"].astype(str)

print("Sentiment shape:", sent.shape)
display(sent.head())


Sentiment shape: (19198, 11)


,ticker,date,sent_mean,sent_std,sent_count,sent_pos,sent_neg,sent_neu,sent_daily_cat,sent_lag1,sent_roll3
0,ABB,2018-01-09,0.997986,0.000000,1,1,0,0,positive,NaN,NaN
1,ABB,2018-01-10,0.977620,0.006999,3,3,0,0,positive,0.997986,0.997986
2,ABB,2018-01-11,0.997827,0.004242,4,4,0,0,positive,0.977620,0.987803
3,ABB,2018-01-12,0.981021,0.018804,5,5,0,0,positive,0.997827,0.991144
4,ABB,2018-01-17,0.983361,0.023424,2,2,0,0,positive,0.981021,0.985489


In [5]:
news_counts = sent.groupby("ticker")["sent_count"].sum().sort_values(ascending=False)
news_counts


ticker
UBS                 31059
SWISSCOM             7170
NESTLE               5631
VONTOBEL             3234
BKW ENERGIE          3214
ABB                  3170
JULIUS BAER          2748
HELVETIA             2254
LONZA                2092
SWATCH               1990
SWISS LIFE           1950
SWISS RE             1789
HOLCIM               1641
ZURICH INSURANCE     1356
EMMI                 1312
RICHEMONT            1273
LOGITECH              973
BALOISE               907
SWISSQUOTE            896
ARYZTA                804
LANDIS                784
GIVAUDAN              773
BLKB                  531
SWISS PRIME SITE      439
ROCHE                 325
NOVARTIS              280
ALCON                 246
SANTHERA               97
ALPINE                 75
REISHAUER              69
Name: sent_count, dtype: int64

In [14]:
sent.ticker.unique()

array(['ABB', 'ALCON', 'ALPINE', 'ARYZTA', 'BALOISE', 'BKW ENERGIE',
       'BLKB', 'EMMI', 'GIVAUDAN', 'HELVETIA', 'HOLCIM', 'JULIUS BAER',
       'LANDIS', 'LOGITECH', 'LONZA', 'NESTLE', 'NOVARTIS', 'REISHAUER',
       'RICHEMONT', 'ROCHE', 'SANTHERA', 'SWATCH', 'SWISS LIFE',
       'SWISS PRIME SITE', 'SWISS RE', 'SWISSCOM', 'SWISSQUOTE', 'UBS',
       'VONTOBEL', 'ZURICH INSURANCE'], dtype=object)

In [15]:
market['ticker'] = market['ticker'].str.upper()

In [7]:
sent["sent_cat_lag1"] = (
    sent.groupby("ticker")["sent_daily_cat"].shift(1)
)

sent = sent[
    [
        "ticker",
        "date",
        "sent_mean",
        "sent_std",
        "sent_count",
        "sent_pos",
        "sent_neg",
        "sent_neu",
        "sent_daily_cat",
        "sent_lag1",
        "sent_cat_lag1",
        "sent_roll3"
    ]
]
display(sent.head())

,ticker,date,sent_mean,sent_std,sent_count,sent_pos,sent_neg,sent_neu,sent_daily_cat,sent_lag1,sent_cat_lag1,sent_roll3
0,ABB,2018-01-09,0.997986,0.000000,1,1,0,0,positive,NaN,NaN,NaN
1,ABB,2018-01-10,0.977620,0.006999,3,3,0,0,positive,0.997986,positive,0.997986
2,ABB,2018-01-11,0.997827,0.004242,4,4,0,0,positive,0.977620,positive,0.987803
3,ABB,2018-01-12,0.981021,0.018804,5,5,0,0,positive,0.997827,positive,0.991144
4,ABB,2018-01-17,0.983361,0.023424,2,2,0,0,positive,0.981021,positive,0.985489


In [8]:
sent.to_parquet("../../data/processed/sentiment_daily_category_shifted.parquet")
sent.to_csv("../../data/processed/sentiment_daily_category_shifted.csv", index=False)

In [16]:
# Merge
features = market.merge(
    sent,
    on=["ticker", "date"],
    how="left",    # keep ALL market rows (important!)
    validate="many_to_one"  # each (ticker, date) in sentiment appears once
)

In [17]:
print("Merged shape:", features.shape)
features.head()

Merged shape: (52454, 31)


,date,open,high,low,close,volume,adj close,ticker,sector,cap,return,log_return,year,vol21,vol63,ret_ema21,ret_ema63,px_ma21,px_ma63,px_ma200,trend200_up,sent_mean,sent_std,sent_count,sent_pos,sent_neg,sent_neu,sent_daily_cat,sent_lag1,sent_cat_lag1,sent_roll3
0,2018-01-03,19.562585,19.749109,19.547663,19.719265,5519259.0,19.719265,ABB,Real Estate - Energy - Industrials,Mid Cap,NaN,NaN,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-04,19.838642,19.958017,19.749111,19.935635,5738092.0,19.935635,ABB,Real Estate - Energy - Industrials,Mid Cap,0.010972,0.010913,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-05,19.920712,20.084852,19.883406,20.084852,4435594.0,20.084852,ABB,Real Estate - Energy - Industrials,Mid Cap,0.007485,0.007457,2018,NaN,NaN,0.010972,0.010972,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-08,20.099777,20.137081,19.920715,20.092316,5029780.0,20.092316,ABB,Real Estate - Energy - Industrials,Mid Cap,0.000372,0.000372,2018,NaN,NaN,0.010655,0.010864,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-09,19.972940,20.196768,19.935636,20.196768,6974533.0,20.196768,ABB,Real Estate - Energy - Industrials,Mid Cap,0.005199,0.005185,2018,NaN,NaN,0.009721,0.010536,NaN,NaN,NaN,0,0.997986,0.0,1.0,1.0,0.0,0.0,positive,NaN,NaN,NaN


In [18]:
news_counts = features.groupby("ticker")["sent_count"].sum().sort_values(ascending=False)
news_counts

ticker
UBS                 30523.0
SWISSCOM             7052.0
NESTLE               5543.0
VONTOBEL             3208.0
BKW ENERGIE          3164.0
ABB                  3140.0
JULIUS BAER          2718.0
HELVETIA             2218.0
LONZA                2066.0
SWATCH               1952.0
SWISS LIFE           1928.0
SWISS RE             1761.0
HOLCIM               1630.0
ZURICH INSURANCE     1319.0
EMMI                 1297.0
RICHEMONT            1264.0
LOGITECH              969.0
BALOISE               898.0
SWISSQUOTE            887.0
ARYZTA                800.0
LANDIS                772.0
GIVAUDAN              770.0
BLKB                  526.0
SWISS PRIME SITE      438.0
ROCHE                 320.0
NOVARTIS              279.0
ALCON                 240.0
SANTHERA               97.0
ALPINE                 73.0
REISHAUER              68.0
Name: sent_count, dtype: float64

In [19]:
# 1. Sort by ticker/date (always good for time series)
features = features.sort_values(["ticker", "date"]).reset_index(drop=True)

# 2. Drop rows where return is NaN (first day per ticker)
before = features.shape[0]
features = features[~features["return"].isna()].copy()
after = features.shape[0]
print(f"Dropped {before - after} rows with NaN return.")

# 3. Create stable row_id
features["row_id"] = features["ticker"] + "|" + features["date"].astype(str)

# 4. Fill sentiment NaNs for no-news days
sent_cols_num = [
    "sent_mean", "sent_std", "sent_count",
    "sent_pos", "sent_neg", "sent_neu",
    "sent_lag1", "sent_roll3",
]

for col in sent_cols_num:
    if col in features.columns:
        features[col] = features[col].fillna(0)

if "sent_daily_cat" in features.columns:
    features["sent_daily_cat"] = features["sent_daily_cat"].fillna("neutral")

features.head()


Dropped 30 rows with NaN return.


,date,open,high,low,close,volume,adj close,ticker,sector,cap,return,log_return,year,vol21,vol63,ret_ema21,ret_ema63,px_ma21,px_ma63,px_ma200,trend200_up,sent_mean,sent_std,sent_count,sent_pos,sent_neg,sent_neu,sent_daily_cat,sent_lag1,sent_cat_lag1,sent_roll3,row_id
1,2018-01-04,19.838642,19.958017,19.749111,19.935635,5738092.0,19.935635,ABB,Real Estate - Energy - Industrials,Mid Cap,0.010972,0.010913,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.000000,0.000000,0.0,0.0,0.0,0.0,neutral,0.000000,NaN,0.000000,ABB|2018-01-04
2,2018-01-05,19.920712,20.084852,19.883406,20.084852,4435594.0,20.084852,ABB,Real Estate - Energy - Industrials,Mid Cap,0.007485,0.007457,2018,NaN,NaN,0.010972,0.010972,NaN,NaN,NaN,0,0.000000,0.000000,0.0,0.0,0.0,0.0,neutral,0.000000,NaN,0.000000,ABB|2018-01-05
3,2018-01-08,20.099777,20.137081,19.920715,20.092316,5029780.0,20.092316,ABB,Real Estate - Energy - Industrials,Mid Cap,0.000372,0.000372,2018,NaN,NaN,0.010655,0.010864,NaN,NaN,NaN,0,0.000000,0.000000,0.0,0.0,0.0,0.0,neutral,0.000000,NaN,0.000000,ABB|2018-01-08
4,2018-01-09,19.972940,20.196768,19.935636,20.196768,6974533.0,20.196768,ABB,Real Estate - Energy - Industrials,Mid Cap,0.005199,0.005185,2018,NaN,NaN,0.009721,0.010536,NaN,NaN,NaN,0,0.997986,0.000000,1.0,1.0,0.0,0.0,positive,0.000000,NaN,0.000000,ABB|2018-01-09
5,2018-01-10,20.144539,20.159461,19.995321,20.055008,4184285.0,20.055008,ABB,Real Estate - Energy - Industrials,Mid Cap,-0.007019,-0.007044,2018,NaN,NaN,0.009309,0.010369,NaN,NaN,NaN,0,0.977620,0.006999,3.0,3.0,0.0,0.0,positive,0.997986,positive,0.997986,ABB|2018-01-10


In [20]:
import pandas as pd
import numpy as np

# 1) Make sure date is datetime and data is sorted
features["date"] = pd.to_datetime(features["date"])
features = features.sort_values(["date", "ticker"]).reset_index(drop=True)

# 2) Choose a clear calendar cutoff for train/test
# 👉 Adjust this if you want a different boundary
split_date = pd.Timestamp("2023-01-01")

# 3) Build masks
train_mask = features["date"] < split_date
test_mask  = features["date"] >= split_date

# 4) Add split columns (these will be reused for ALL models)
features["is_train"] = train_mask
features["set"] = np.where(features["is_train"], "train", "test")

# 5) Quick sanity checks
print(features["set"].value_counts())
print(features.groupby("set")["date"].agg(["min", "max"]))


set
train    37394
test     15030
Name: count, dtype: int64
             min        max
set                        
test  2023-01-03 2024-12-30
train 2018-01-04 2022-12-30


In [21]:
features.head()

,date,open,high,low,close,volume,adj close,ticker,sector,cap,return,log_return,year,vol21,vol63,ret_ema21,ret_ema63,px_ma21,px_ma63,px_ma200,trend200_up,sent_mean,sent_std,sent_count,sent_pos,sent_neg,sent_neu,sent_daily_cat,sent_lag1,sent_cat_lag1,sent_roll3,row_id,is_train,set
0,2018-01-04,19.838642,19.958017,19.749111,19.935635,5738092.0,19.935635,ABB,Real Estate - Energy - Industrials,Mid Cap,0.010972,0.010913,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.000000,0.0,0.0,0.0,0.0,0.0,neutral,0.0,NaN,0.000000,ABB|2018-01-04,True,train
1,2018-01-04,7.882366,7.882366,7.882366,7.882366,0.0,7.882366,ALPINE,Consumer Discretionary,Large Cap,0.000000,0.000000,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.000000,0.0,0.0,0.0,0.0,0.0,neutral,0.0,NaN,0.000000,ALPINE|2018-01-04,True,train
2,2018-01-04,332.666046,333.091339,317.440460,323.224487,111998.0,323.224487,ARYZTA,Consumer Staples,Mid Cap,-0.024390,-0.024693,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.946913,0.0,1.0,1.0,0.0,0.0,positive,0.0,NaN,0.999941,ARYZTA|2018-01-04,True,train
3,2018-01-04,107.037628,107.741360,106.545007,107.670982,111416.0,107.670982,BALOISE,Insurance,Mid Cap,0.009235,0.009192,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.000000,0.0,0.0,0.0,0.0,0.0,neutral,0.0,NaN,0.000000,BALOISE|2018-01-04,True,train
4,2018-01-04,47.071196,47.562372,47.071196,47.480511,19374.0,47.480511,BKW ENERGIE,Real Estate - Energy - Industrials,Small Cap,0.003460,0.003454,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.041373,0.0,1.0,0.0,0.0,1.0,neutral,0.0,NaN,0.995683,BKW ENERGIE|2018-01-04,True,train


In [22]:
features.to_parquet("../../data/processed/market_all_features_with_sentiment.parquet")
features.to_csv("../../data/processed/market_all_features_with_sentiment.csv", index=False)